# Ajustar no treino e aplicar a regra

A distância de uma viagem de validação está ausente. Você tem alguns valores no treino e um valor alto na validação. De quais observações virá o preenchimento?

Retome [Treino, validação e teste](https://devrgs.github.io/Aprendendo-linguagens/#/python/preparacao-ml/particoes) e [Precisão e escalas](https://devrgs.github.io/Aprendendo-linguagens/#/python/matematica/precisao-e-escalas). Os exercícios usam a biblioteca padrão, guardam a saída do último em JSON e criam os dados de entrada no código.

Ajustar é calcular parâmetros a partir dos exemplos de treino. Aplicar é preencher e transformar com esses valores já calculados. A mediana dos valores observados no treino preenche ausentes nas três partições. Para escala mínimo-máximo, subtraia o mínimo do treino e divida pela amplitude do treino. Se a amplitude for zero, esta aula define escalados como zero. Valores novos podem ficar fora de [0, 1].

Execute as células na ordem com **Shift+Enter**. A preparação cria uma pasta nova na área de arquivos do caderno. Baixe o caderno editado e os resultados que quiser guardar.


In [ ]:
from pathlib import Path as _Path
from tempfile import mkdtemp as _mkdtemp
import os as _os
import sys as _sys
from importlib import invalidate_caches as _refresh_imports

# Auxiliares editáveis devem ser relidos sem aproveitar bytecode antigo.
_sys.dont_write_bytecode = True

_raiz = _Path.cwd()
_trabalho = _Path(_mkdtemp(prefix='python-ajustar-e-aplicar-', dir=_raiz))
import numpy
import pandas


## E1 · Preveja e execute

Ache a mediana dos valores observados em [2, 4, ausente, 6].


In [ ]:
_pasta = _trabalho / 'e1'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_refresh_imports()


In [ ]:
from statistics import median

treino = [2, 4, None, 6]
observados = [km for km in treino if km is not None]
print(median(observados))


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Saída esperada

```text
4
```

A mediana usa três números observados. A célula ausente não entra no cálculo.


## E2 · Altere ou depure

A validação é [100, ausente]. Compare preencher com 4, da mediana do treino, e com a mediana local.


In [ ]:
_pasta = _trabalho / 'e2'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_refresh_imports()


In [ ]:
from statistics import median

validacao = [100, None]
local = median(km for km in validacao if km is not None)
for valor in (4, local):
    print([valor if km is None else km for km in validacao])


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Saída esperada

```text
[100, 4]
[100, 100]
```

O preenchimento com a mediana local aprende sobre o conjunto reservado. Reuse a mediana do treino.


## E3 · Resolva

Implemente ajustar e aplicar separadamente. Salve e recarregue parâmetros de [2,4,ausente,6]. Troque somente um valor de teste e confirme que o JSON guardado se mantém. Cubra amplitude zero e treino todo ausente.


In [ ]:
_pasta = _trabalho / 'e3'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_refresh_imports()


In [ ]:
import json
import math
from pathlib import Path
from statistics import median

def ajustar(treino):
    observados = [km for km in treino if km is not None]
    if not observados:
        raise ValueError("treino sem km observados")
    return {"mediana": median(observados), "minimo": min(observados), "maximo": max(observados)}

def aplicar(lote, params):
    amplitude = params["maximo"] - params["minimo"]
    preenchidos = [params["mediana"] if km is None else km for km in lote]
    return [(km - params["minimo"]) / amplitude if amplitude else 0.0 for km in preenchidos]

treino = [2, 4, None, 6]
caminho = Path("parametros.json")
caminho.write_text(json.dumps(ajustar(treino), sort_keys=True), encoding="utf-8")
antes = caminho.read_text(encoding="utf-8")
params = json.loads(antes)
print(aplicar([None, 8], params))
print(aplicar([None, 12], params))
assert json.dumps(ajustar(treino), sort_keys=True) == antes
assert caminho.read_text(encoding="utf-8") == antes
assert math.isclose(aplicar([None], params)[0], 0.5, abs_tol=1e-6)
assert aplicar([None, 9], ajustar([4, None, 4])) == [0.0, 0.0]
try:
    ajustar([None])
except ValueError as erro:
    print(erro)


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Saída esperada

```text
[0.5, 1.5]
[0.5, 2.5]
treino sem km observados
```

A troca de km altera apenas o escalado da segunda linha de teste. A min-max do treino pode produzir 2,5 e não recorta valores. Uma coluna constante mapeia todos os valores para zero pela regra explícita. A próxima aula usa o alvo de treino para uma previsão de referência.
